# Evaluate CodeBERT on balanced `setup_py_dataset`

Input on Google Drive:
- `NT230/data/d1/saved_models/checkpoint-best-acc/model.bin`
- `NT230/data/setup_py_dataset_balanced.zip`

Expected zip content:
- `setup_py_dataset/benign/*/setup.py`
- `setup_py_dataset/malicious/*/setup.py`

This notebook does not use LLM agents. It evaluates the fine-tuned CodeBERT model directly on each `setup.py` sample.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, json, zipfile, shutil
from pathlib import Path
from collections import Counter

DRIVE_ROOT = Path('/content/drive/My Drive/NT230')
DRIVE_DATA = DRIVE_ROOT / 'data'
MODEL_PATH = DRIVE_DATA / 'd1/saved_models/checkpoint-best-acc/model.bin'

# Prefer the balanced zip created locally: D:\lamps-jss\data\setup_py_dataset_balanced.zip
BALANCED_ZIP = DRIVE_DATA / 'setup_py_dataset_balanced.zip'
LEGACY_ZIP = DRIVE_DATA / 'setup_py_dataset.zip'
DRIVE_DATASET_DIR = DRIVE_DATA / 'setup_py_dataset'
LOCAL_DATASET_PARENT = Path('/content/evaluate_newdataset_data')
DATASET_DIR = LOCAL_DATASET_PARENT / 'setup_py_dataset'
OUTPUT_DIR = DRIVE_DATA / 'setup_py_dataset_balanced_results'


def has_labels(path: Path) -> bool:
    return (path / 'benign').exists() and (path / 'malicious').exists()


def setup_count(path: Path, label: str) -> int:
    label_dir = path / label
    if not label_dir.exists():
        return 0
    return sum(1 for p in label_dir.rglob('setup.py') if p.is_file())


def extract_dataset(zip_path: Path) -> None:
    if LOCAL_DATASET_PARENT.exists():
        shutil.rmtree(LOCAL_DATASET_PARENT)
    LOCAL_DATASET_PARENT.mkdir(parents=True, exist_ok=True)
    print('Extracting:', zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(LOCAL_DATASET_PARENT)


if BALANCED_ZIP.exists():
    extract_dataset(BALANCED_ZIP)
elif LEGACY_ZIP.exists():
    print('Balanced zip not found; falling back to legacy zip:', LEGACY_ZIP)
    extract_dataset(LEGACY_ZIP)
elif has_labels(DRIVE_DATASET_DIR):
    print('Zip not found; falling back to Drive folder:', DRIVE_DATASET_DIR)
    DATASET_DIR = DRIVE_DATASET_DIR
    OUTPUT_DIR = DATASET_DIR / 'results_codebert'
else:
    raise FileNotFoundError(f'Missing dataset. Expected {BALANCED_ZIP} or {DRIVE_DATASET_DIR}')

# Handle nested zip layouts.
if not has_labels(DATASET_DIR):
    candidates = [p for p in LOCAL_DATASET_PARENT.rglob('setup_py_dataset') if has_labels(p)]
    if candidates:
        DATASET_DIR = candidates[0]

assert MODEL_PATH.exists(), f'Missing model: {MODEL_PATH}'
assert has_labels(DATASET_DIR), f'Missing benign/malicious folders under: {DATASET_DIR}'

counts_preview = {
    'benign': setup_count(DATASET_DIR, 'benign'),
    'malicious': setup_count(DATASET_DIR, 'malicious'),
}
print('Model:', MODEL_PATH)
print('Dataset:', DATASET_DIR)
print('Zip:', BALANCED_ZIP if BALANCED_ZIP.exists() else LEGACY_ZIP if LEGACY_ZIP.exists() else 'folder')
print('Counts:', counts_preview)
assert counts_preview['benign'] > 0 and counts_preview['malicious'] > 0, 'No setup.py files found for one label'


In [ ]:
!pip install -q transformers==4.40.0 torch scikit-learn scipy pandas tqdm

In [ ]:
REPO_DIR = Path('/content/NT230')
if not REPO_DIR.exists():
    !git clone --depth=1 https://github.com/khoilv2005/NT230.git /content/NT230
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo ready:', REPO_DIR)

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')

In [ ]:
from lamps.agents.classifier import ClassifierAgent
from lamps.agents.extractor import ExtractedFile
from lamps.evaluation.metrics import classification_report, format_report

classifier = ClassifierAgent(
    checkpoint=str(MODEL_PATH),
    batch_size=64,
)
print('Classifier loaded')

In [ ]:
LABELS = {'benign': 0, 'malicious': 1}
records = []

for label_name, target in LABELS.items():
    label_dir = DATASET_DIR / label_name
    for setup_path in sorted(label_dir.rglob('setup.py')):
        sample_id = setup_path.parent.name
        source = setup_path.read_text(encoding='utf-8', errors='ignore')
        if not source.strip():
            continue
        records.append({
            'sample_id': sample_id,
            'label_name': label_name,
            'target': target,
            'path': setup_path,
            'source': source,
        })

counts = Counter(r['label_name'] for r in records)
print('Samples:', len(records))
print('Counts:', dict(counts))
assert records, 'No setup.py records found'

In [ ]:
from tqdm import tqdm

files = [
    ExtractedFile(
        package=r['sample_id'],
        path=r['path'],
        rel_path='setup.py',
        source=r['source'],
    )
    for r in records
]

print(f'Classifying {len(files)} setup.py files...')
classifications = classifier.classify_files(files)

y_true = [int(r['target']) for r in records]
y_pred = [int(c.target) for c in classifications]
report = classification_report(y_true, y_pred)

print('\n=== setup_py_dataset_balanced / CodeBERT ===')
print(format_report(report))


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

predictions = []
for r, c in zip(records, classifications):
    predictions.append({
        'sample_id': r['sample_id'],
        'path': str(r['path'].relative_to(DATASET_DIR)),
        'target': int(r['target']),
        'target_label': r['label_name'],
        'predicted': int(c.target),
        'predicted_label': c.label,
        'score': float(c.score),
        'source_chars': len(r['source']),
    })

(OUTPUT_DIR / 'predictions.jsonl').write_text(
    '\n'.join(json.dumps(p, ensure_ascii=False) for p in predictions) + '\n',
    encoding='utf-8',
)
(OUTPUT_DIR / 'report.json').write_text(json.dumps(report.to_dict(), indent=2), encoding='utf-8')
(OUTPUT_DIR / 'report.txt').write_text(format_report(report), encoding='utf-8')
(OUTPUT_DIR / 'summary.json').write_text(json.dumps({
    'dataset': str(DATASET_DIR),
    'model': str(MODEL_PATH),
    'zip': str(BALANCED_ZIP if BALANCED_ZIP.exists() else LEGACY_ZIP if LEGACY_ZIP.exists() else ''),
    'counts': dict(counts),
    'n_samples': len(records),
    'output': str(OUTPUT_DIR),
}, indent=2), encoding='utf-8')

print('Saved to:', OUTPUT_DIR)
print('- predictions.jsonl')
print('- report.json')
print('- report.txt')
print('- summary.json')


In [ ]:
# Export wrong predictions: FP/FN for manual inspection
import csv

manifest_by_id = {}
manifest_path = DATASET_DIR / 'manifest.csv'
if manifest_path.exists():
    with manifest_path.open(newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            manifest_by_id[row.get('sample_id', '')] = row

wrong = []
false_positives = []  # benign -> malicious
false_negatives = []  # malicious -> benign

for p in predictions:
    if int(p['target']) == int(p['predicted']):
        continue
    meta = manifest_by_id.get(p['sample_id'], {})
    row = {
        **p,
        'error_type': 'FP' if int(p['target']) == 0 and int(p['predicted']) == 1 else 'FN',
        'dataset_source': meta.get('dataset', ''),
        'source_archive': meta.get('source_archive', ''),
        'setup_member': meta.get('setup_member', ''),
        'output': meta.get('output', ''),
    }
    wrong.append(row)
    if row['error_type'] == 'FP':
        false_positives.append(row)
    else:
        false_negatives.append(row)

def write_jsonl(path, rows):
    path.write_text(
        '\n'.join(json.dumps(r, ensure_ascii=False) for r in rows) + ('\n' if rows else ''),
        encoding='utf-8',
    )

def write_csv(path, rows):
    fieldnames = [
        'sample_id', 'error_type', 'target', 'target_label', 'predicted', 'predicted_label',
        'score', 'source_chars', 'path', 'dataset_source', 'source_archive', 'setup_member', 'output',
    ]
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

# Sort for easier inspection:
# - FN first by lowest malicious score (most confidently missed)
# - FP first by highest malicious score (most confidently false alarm)
false_negatives_sorted = sorted(false_negatives, key=lambda r: float(r['score']))
false_positives_sorted = sorted(false_positives, key=lambda r: -float(r['score']))
wrong_sorted = false_negatives_sorted + false_positives_sorted

write_jsonl(OUTPUT_DIR / 'wrong_predictions.jsonl', wrong_sorted)
write_jsonl(OUTPUT_DIR / 'false_negatives.jsonl', false_negatives_sorted)
write_jsonl(OUTPUT_DIR / 'false_positives.jsonl', false_positives_sorted)
write_csv(OUTPUT_DIR / 'wrong_predictions.csv', wrong_sorted)
write_csv(OUTPUT_DIR / 'false_negatives.csv', false_negatives_sorted)
write_csv(OUTPUT_DIR / 'false_positives.csv', false_positives_sorted)

error_summary = {
    'wrong': len(wrong),
    'false_positives': len(false_positives),
    'false_negatives': len(false_negatives),
    'wrong_predictions_csv': str(OUTPUT_DIR / 'wrong_predictions.csv'),
    'false_negatives_csv': str(OUTPUT_DIR / 'false_negatives.csv'),
    'false_positives_csv': str(OUTPUT_DIR / 'false_positives.csv'),
}
(OUTPUT_DIR / 'error_summary.json').write_text(json.dumps(error_summary, indent=2), encoding='utf-8')

print(json.dumps(error_summary, indent=2))
print('\nTop 10 false negatives (malware missed):')
for r in false_negatives_sorted[:10]:
    print(f"FN score={float(r['score']):.4f} sample={r['sample_id']} source={r['dataset_source']} path={r['path']}")

print('\nTop 10 false positives (benign false alarm):')
for r in false_positives_sorted[:10]:
    print(f"FP score={float(r['score']):.4f} sample={r['sample_id']} source={r['dataset_source']} path={r['path']}")
